# 00_Data ingest and sanity checks
Converts the July 2025 Citi Bike trip files to Parquet, profiles their size and schema, cleans trips, defines the study area and its station capacities, and checks the net-flow pattern that motivates overnight rebalancing.

## 1. Load data

Converts each month's raw CSV files to one Parquet file in `data/processed/`. The conversion is skipped automatically if the Parquet file already exists. Add a month to `months` to load more data.

| Name | Source | Role |
|---|---|---|
| `trips_raw` | `data/raw/<yyyymm>/*.csv` (Citi Bike System Data) | One row per trip |
| `stations` | `data/external/station_information_20261101.json` (GBFS) | Station capacity and location, snapshot taken 2026-11-01 |

In [16]:
import os
import duckdb

duckdb.sql("SET enable_progress_bar = false")
duckdb.sql("SET memory_limit = '4GB'")  # 8GB machine; leave room for the OS
raw_dir = '../data/raw'
external_dir = '../data/external'
processed_dir = '../data/processed'
os.makedirs(processed_dir, exist_ok=True)

months = ['202507']

# Skip if the Parquet file already exists; convert only on the first run
for month in months:
    parquet_path = f'{processed_dir}/trips_{month}.parquet'
    if os.path.exists(parquet_path):
        print(f'{month}: Parquet already exists, skipping conversion.')
    else:
        duckdb.sql(f"""
        COPY (
            SELECT *
            FROM read_csv('{raw_dir}/{month}/*.csv',
                          union_by_name = true,
                          types = {{'start_station_id': 'VARCHAR',
                                   'end_station_id':   'VARCHAR'}})
        )
        TO '{parquet_path}' (FORMAT PARQUET)
        """)
        print(f'{month}: conversion done.')

202507: conversion done.


### Register the `trips_raw` and `stations` views
`trips_raw` reads every monthly Parquet file. `stations` flattens the GBFS JSON to one row per station; trip data uses the GBFS `short_name` (e.g. `6140.05`) as its station ID, so `short_name` becomes `station_id`. Re-run this cell after a kernel restart.

In [17]:
duckdb.sql(f"""
CREATE OR REPLACE VIEW trips_raw AS
SELECT * FROM '{processed_dir}/trips_*.parquet';

CREATE OR REPLACE VIEW stations AS
SELECT s.short_name AS station_id,
       s.name,
       s.lat,
       s.lon AS lng,
       s.capacity
FROM (
    SELECT unnest(data.stations) AS s
    FROM read_json('{external_dir}/station_information_20261101.json')
);
""")

## 2. Data profiling

### Schema check
Station IDs are read as `VARCHAR` so values like `6140.05` keep their format.

In [18]:
duckdb.sql("DESCRIBE SELECT * FROM trips_raw").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ ride_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ rideable_type      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ started_at         │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ ended_at           │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ start_station_name │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ start_station_id   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ end_station_name   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ end_station_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ start_lat          │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │

### Row count and date range

In [19]:
duckdb.sql("""
SELECT COUNT(*)        AS n_trips,
       MIN(started_at) AS first_start,
       MAX(started_at) AS last_start
FROM trips_raw
""").show()

┌─────────┬─────────────────────────┬─────────────────────────┐
│ n_trips │       first_start       │       last_start        │
│  int64  │        timestamp        │        timestamp        │
├─────────┼─────────────────────────┼─────────────────────────┤
│ 4988053 │ 2025-06-29 23:16:34.475 │ 2025-07-31 23:57:27.342 │
└─────────┴─────────────────────────┴─────────────────────────┘



### Missing stations
Trips with no start or end station, mostly e-bikes not returned to a dock.

In [20]:
duckdb.sql("""
SELECT COUNT(*) FILTER (WHERE start_station_id IS NULL) AS null_start,
       COUNT(*) FILTER (WHERE end_station_id IS NULL)   AS null_end,
       ROUND(100.0 * COUNT(*) FILTER (WHERE start_station_id IS NULL) / COUNT(*), 2) AS null_start_pct,
       ROUND(100.0 * COUNT(*) FILTER (WHERE end_station_id IS NULL)   / COUNT(*), 2) AS null_end_pct
FROM trips_raw
""").show()

┌────────────┬──────────┬────────────────┬──────────────┐
│ null_start │ null_end │ null_start_pct │ null_end_pct │
│   int64    │  int64   │     double     │    double    │
├────────────┼──────────┼────────────────┼──────────────┤
│       2997 │    20399 │           0.06 │         0.41 │
└────────────┴──────────┴────────────────┴──────────────┘



### Bike type and rider type
`SUM(COUNT(*)) OVER ()` gives the overall total as the denominator.

In [21]:
duckdb.sql("""
SELECT rideable_type,
       member_casual,
       COUNT(*) AS n,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM trips_raw
GROUP BY rideable_type, member_casual
ORDER BY n DESC
""").show()

┌───────────────┬───────────────┬─────────┬────────┐
│ rideable_type │ member_casual │    n    │  pct   │
│    varchar    │    varchar    │  int64  │ double │
├───────────────┼───────────────┼─────────┼────────┤
│ electric_bike │ member        │ 2738273 │   54.9 │
│ classic_bike  │ member        │ 1194479 │   23.9 │
│ electric_bike │ casual        │  792538 │   15.9 │
│ classic_bike  │ casual        │  262763 │    5.3 │
└───────────────┴───────────────┴─────────┴────────┘



### Trip duration
1st, 50th and 99th percentiles in minutes, used to set the cleaning thresholds.

In [22]:
duckdb.sql("""
SELECT quantile_cont(date_diff('minute', started_at, ended_at), [0.01, 0.5, 0.99]) AS duration_min_q
FROM trips_raw
""").show()

┌───────────────────┐
│  duration_min_q   │
│     double[]      │
├───────────────────┤
│ [1.0, 10.0, 66.0] │
└───────────────────┘



### Station count

In [23]:
duckdb.sql("""
SELECT COUNT(DISTINCT start_station_id) AS n_stations
FROM trips_raw
""").show()

┌────────────┐
│ n_stations │
│   int64    │
├────────────┤
│       2157 │
└────────────┘



**Key findings: data profiling**
- The July 2025 files hold 4,988,053 trips. The earliest trip starts on 2025-06-29, so files are grouped by end time and a few trips cross months.
- Missing station IDs are rare: 2,997 start (0.06%) and 20,399 end (0.41%).
- E-bikes account for 70.8% of trips and members for 78.8%.
- The median trip lasts 10 minutes; the 1st and 99th percentiles are 1 and 66 minutes.
- Trips start at 2,157 stations across the system.

## 3. Cleaning

### Register the `trips_clean` view
Keeps trips that start within the study period and last 1 minute to 3 hours. Trips under 1 minute are usually re-docks; trips over 3 hours are usually unreturned bikes. Trips with a missing end station are kept, because their departure is still real demand at the start station; arrival counts exclude them later.

In [24]:
start_date, end_date = '2025-07-01', '2025-08-01'

duckdb.sql(f"""
CREATE OR REPLACE VIEW trips_clean AS
SELECT *,
       date_diff('second', started_at, ended_at) / 60.0 AS duration_min
FROM trips_raw
WHERE started_at >= '{start_date}'
  AND started_at <  '{end_date}'
  AND date_diff('second', started_at, ended_at) BETWEEN 60 AND 3 * 3600
""")

### Rows removed

In [25]:
duckdb.sql("""
WITH counts AS (
    SELECT (SELECT COUNT(*) FROM trips_raw)   AS n_raw,
           (SELECT COUNT(*) FROM trips_clean) AS n_clean
)
SELECT n_raw,
       n_clean,
       ROUND(100.0 * (1 - n_clean / n_raw), 2) AS removed_pct
FROM counts
""").show()

┌─────────┬─────────┬─────────────┐
│  n_raw  │ n_clean │ removed_pct │
│  int64  │  int64  │   double    │
├─────────┼─────────┼─────────────┤
│ 4988053 │ 4977167 │        0.22 │
└─────────┴─────────┴─────────────┘



### GBFS match rate
Share of cleaned trips whose start station appears in the GBFS snapshot. The snapshot is from November 2026, so stations removed or renumbered since July 2025 do not match.

In [26]:
duckdb.sql("""
SELECT ROUND(100.0 * COUNT(s.station_id) / COUNT(*), 1) AS match_pct
FROM trips_clean t
LEFT JOIN stations s
       ON t.start_station_id = s.station_id
""").show()

┌───────────┐
│ match_pct │
│  double   │
├───────────┤
│      96.4 │
└───────────┘



**Key findings: cleaning**
- Cleaning keeps 4,977,167 trips and removes 0.22% (cross-month starts, trips under 1 minute or over 3 hours).
- 96.4% of cleaned trips start at a station in the GBFS snapshot; the remaining stations need a capacity fallback.

## 4. Study area

### Station locations
One row per start station. Location is the median start coordinate, so stations no longer in GBFS are still placed.

In [27]:
duckdb.sql("""
CREATE OR REPLACE TABLE station_loc AS
SELECT start_station_id              AS station_id,
       any_value(start_station_name) AS name,
       median(start_lat)             AS lat,
       median(start_lng)             AS lng,
       COUNT(*)                      AS n_departures
FROM trips_clean
WHERE start_station_id IS NOT NULL
GROUP BY start_station_id
""")

### Candidate areas
Midtown alone is mostly offices. Extending south to 14th St adds residential areas (Chelsea, Murray Hill, Kips Bay), so the area contains both morning outflow and inflow stations.

In [28]:
duckdb.sql("""
WITH boxes (area, lat_min, lat_max, lng_min, lng_max) AS (
    VALUES ('Midtown (34th-59th St)', 40.745, 40.770, -74.010, -73.965),
           ('14th-59th St',           40.730, 40.770, -74.012, -73.965)
)
SELECT b.area,
       COUNT(*)            AS n_stations,
       SUM(l.n_departures) AS n_departures
FROM boxes b
JOIN station_loc l
  ON l.lat BETWEEN b.lat_min AND b.lat_max
 AND l.lng BETWEEN b.lng_min AND b.lng_max
GROUP BY b.area
ORDER BY n_stations
""").show()

┌────────────────────────┬────────────┬──────────────┐
│          area          │ n_stations │ n_departures │
│        varchar         │   int64    │    int128    │
├────────────────────────┼────────────┼──────────────┤
│ Midtown (34th-59th St) │        135 │       871700 │
│ 14th-59th St           │        227 │      1518949 │
└────────────────────────┴────────────┴──────────────┘



### Register `study_stations` with capacity
Keeps stations in the 14th–59th St box. Capacity comes from the GBFS ID match first, then a name match for renumbered stations, then the area median. `capacity_source` records which rule was used.

In [29]:
duckdb.sql("""
CREATE OR REPLACE TABLE study_stations AS
WITH area AS (
    SELECT *
    FROM station_loc
    WHERE lat BETWEEN 40.730 AND 40.770
      AND lng BETWEEN -74.012 AND -73.965
),
by_name AS (
    -- capacity looked up by station name, for stations whose ID changed
    SELECT name, any_value(capacity) AS cap_by_name
    FROM stations
    GROUP BY name
),
joined AS (
    SELECT a.*,
           s.capacity    AS cap_by_id,
           n.cap_by_name
    FROM area a
    LEFT JOIN stations s ON a.station_id = s.station_id
    LEFT JOIN by_name n  ON a.name = n.name
)
SELECT station_id,
       name,
       lat,
       lng,
       n_departures,
       COALESCE(cap_by_id, cap_by_name,
                (SELECT median(cap_by_id) FROM joined))::INT AS capacity,
       CASE WHEN cap_by_id   IS NOT NULL THEN 'gbfs_id'
            WHEN cap_by_name IS NOT NULL THEN 'gbfs_name'
            ELSE 'imputed_median'
       END AS capacity_source
FROM joined
""")

### Capacity source

In [30]:
duckdb.sql("""
SELECT capacity_source,
       COUNT(*)          AS n_stations,
       SUM(n_departures) AS n_departures,
       ROUND(100.0 * SUM(n_departures) / SUM(SUM(n_departures)) OVER (), 1) AS departures_pct
FROM study_stations
GROUP BY capacity_source
ORDER BY n_stations DESC
""").show()

┌─────────────────┬────────────┬──────────────┬────────────────┐
│ capacity_source │ n_stations │ n_departures │ departures_pct │
│     varchar     │   int64    │    int128    │     double     │
├─────────────────┼────────────┼──────────────┼────────────────┤
│ gbfs_id         │        210 │      1435688 │           94.5 │
│ imputed_median  │         14 │        78982 │            5.2 │
│ gbfs_name       │          3 │         4279 │            0.3 │
└─────────────────┴────────────┴──────────────┴────────────────┘



**Key findings: study area**
- Midtown alone has 135 stations. The 14th–59th St area has 227 stations and 1,518,949 departures, and is used from here on.
- Capacity: 210 stations matched by GBFS ID, 3 by name, 14 imputed with the area median. Imputed stations carry 78,982 departures (5.2%), so the optimization needs a capacity sensitivity check.

## 5. Net flow patterns

### Weekday count
Number of weekdays in the study period, used to turn totals into daily averages. July 4 is counted as a weekday here; holidays are handled as a feature in forecasting.

In [31]:
n_weekdays = duckdb.sql("""
SELECT COUNT(DISTINCT started_at::DATE)
FROM trips_clean
WHERE dayofweek(started_at) BETWEEN 1 AND 5
""").fetchall()[0][0]
n_weekdays

23

### Morning peak station types
Average weekday net flow (arrivals − departures) per station from 7 to 10 AM. Stations below −5 bikes per day are outflow stations; stations above +5 are inflow stations.

In [32]:
duckdb.sql(f"""
WITH am AS (
    -- departures count as -1, arrivals as +1
    SELECT start_station_id AS station_id, -1 AS flow
    FROM trips_clean
    WHERE hour(started_at) BETWEEN 7 AND 9
      AND dayofweek(started_at) BETWEEN 1 AND 5
    UNION ALL
    SELECT end_station_id, 1
    FROM trips_clean
    WHERE hour(ended_at) BETWEEN 7 AND 9
      AND dayofweek(ended_at) BETWEEN 1 AND 5
      AND end_station_id IS NOT NULL
),
net AS (
    SELECT station_id,
           SUM(flow) / {n_weekdays} AS avg_daily_net
    FROM am
    WHERE station_id IN (SELECT station_id FROM study_stations)
    GROUP BY station_id
)
SELECT COUNT(*) FILTER (WHERE avg_daily_net < -5)          AS outflow_stations,
       COUNT(*) FILTER (WHERE avg_daily_net > 5)           AS inflow_stations,
       COUNT(*) FILTER (WHERE avg_daily_net BETWEEN -5 AND 5) AS balanced_stations,
       ROUND(SUM(avg_daily_net), 1)                        AS area_total_net
FROM net
""").show()

┌──────────────────┬─────────────────┬───────────────────┬────────────────┐
│ outflow_stations │ inflow_stations │ balanced_stations │ area_total_net │
│      int64       │      int64      │       int64       │     double     │
├──────────────────┼─────────────────┼───────────────────┼────────────────┤
│               75 │             110 │                42 │         2198.5 │
└──────────────────┴─────────────────┴───────────────────┴────────────────┘



### Daily net flow cycle
Average weekday net flow for the whole area by hour. `cum_net` is the running total from midnight.

In [33]:
duckdb.sql(f"""
WITH flows AS (
    SELECT hour(started_at) AS hr, -1 AS flow
    FROM trips_clean
    WHERE dayofweek(started_at) BETWEEN 1 AND 5
      AND start_station_id IN (SELECT station_id FROM study_stations)
    UNION ALL
    SELECT hour(ended_at), 1
    FROM trips_clean
    WHERE dayofweek(ended_at) BETWEEN 1 AND 5
      AND end_station_id IN (SELECT station_id FROM study_stations)
)
SELECT hr,
       ROUND(SUM(flow) / {n_weekdays}, 1)                        AS avg_net,
       ROUND(SUM(SUM(flow)) OVER (ORDER BY hr) / {n_weekdays}, 1) AS cum_net
FROM flows
GROUP BY hr
ORDER BY hr
""").show(max_rows=24)

┌───────┬─────────┬─────────┐
│  hr   │ avg_net │ cum_net │
│ int64 │ double  │ double  │
├───────┼─────────┼─────────┤
│     0 │   -32.5 │   -32.5 │
│     1 │   -19.9 │   -52.4 │
│     2 │   -10.7 │   -63.1 │
│     3 │     3.2 │   -60.0 │
│     4 │    -3.8 │   -63.7 │
│     5 │   -24.9 │   -88.6 │
│     6 │    38.2 │   -50.4 │
│     7 │   327.7 │   277.3 │
│     8 │   887.5 │  1164.8 │
│     9 │   983.3 │  2148.1 │
│    10 │   276.0 │  2424.1 │
│    11 │    33.7 │  2457.8 │
│    12 │    10.6 │  2468.3 │
│    13 │   -56.7 │  2411.6 │
│    14 │  -111.1 │  2300.5 │
│    15 │  -173.6 │  2126.9 │
│    16 │  -609.5 │  1517.4 │
│    17 │ -1124.3 │   393.2 │
│    18 │  -534.7 │  -141.5 │
│    19 │  -206.7 │  -348.2 │
│    20 │  -102.7 │  -450.9 │
│    21 │  -106.3 │  -557.2 │
│    22 │   -74.3 │  -631.5 │
│    23 │   -49.6 │  -681.0 │
└───────┴─────────┴─────────┘
  24 rows         3 columns



**Key findings: net flow patterns**
- Weekday mornings (7–10 AM): 75 outflow, 110 inflow and 42 balanced stations, with a net gain of 2,198.5 bikes for the area. Rebalancing within the area is possible, but the binding morning constraint is free docks at inflow stations.
- Daily cycle: net inflow peaks at +2,468 bikes around noon, then reverses, with the largest hourly outflow at 5 PM (−1,124 bikes).
- The area ends each weekday at −681 bikes. Trip data excludes the operator's own truck moves, so the operator must resupply this area; the optimization model needs an external depot node.
- From midnight to 6 AM hourly net flow stays within ±40 bikes, a natural overnight rebalancing window.

## 6. Save outputs

### Save `study_stations`
Writes the station list to Parquet for the forecasting and optimization notebooks.

In [34]:
duckdb.sql(f"""
COPY study_stations TO '{processed_dir}/study_stations.parquet' (FORMAT PARQUET)
""")